# 05 · Figuras y evidencias para la defensa de Pulso IA

Este cuaderno complementa 01–04 sin reentrenar. Exporta PNG de alta resolución y SVG, y recupera los parámetros reales de las corridas. Usar la misma carpeta de Drive. Si faltan historiales, metadatos o predicciones, se informa y se omite esa figura: no se fabrican valores.

Las matrices de referencia incluidas se transcribieron de las imágenes compartidas. Si hay CSV actuales compatibles, pueden sustituirlas. Las imágenes del dataset son ilustraciones de procesamiento, no una nueva evaluación del clasificador.


In [ ]:
%pip -q install numpy pandas scipy matplotlib pyarrow
from pathlib import Path
from google.colab import drive, files
from IPython.display import display, Image
import json, zipfile, gc
import pandas as pd
drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/TP_IA')  # Editar, igual que en 01–04.
RESULTS = PROJECT_DIR / 'results_colab'
PREPARED = RESULTS / 'prepared'
MODELS = RESULTS / 'models'
OUT = RESULTS / 'visuales_defensa'
OUT.mkdir(parents=True, exist_ok=True)
STAGES = [1, 2]
USE_CURRENT_PREDICTIONS = True
print('Salida:', OUT)


## Funciones y matrices de referencia
F1 y accuracy se pueden recalcular desde las matrices. AUC y los histogramas probabilísticos requieren probabilidades individuales; no se reconstruyen a partir de una matriz.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

TEAL, GOLD, INK = '#008578', '#F3B820', '#15263F'
plt.rcParams.update({'font.family':'DejaVu Sans', 'font.size':12, 'axes.spines.top':False, 'axes.spines.right':False, 'axes.labelcolor':INK, 'text.color':INK, 'axes.titleweight':'bold', 'figure.facecolor':'white', 'savefig.facecolor':'white'})
CLASSES={1:['Reposo','Actividad'],2:['Bradicardia','Taquicardia','FA','Normal']}
MATRICES={
 1:{'RL base':[[15,28],[0,173]],'RF base':[[6,37],[0,173]],'ResNet base':[[33,10],[7,166]],'RF GS':[[24,19],[3,170]],'RL GS':[[25,18],[5,168]],'ResNet 03':[[41,2],[23,150]]},
 2:{'RL base':[[3052,8,192,60],[9,1383,19,40],[27,13,310,6],[196,75,87,1266]],'RF base':[[3276,3,13,20],[7,1428,5,11],[47,33,248,28],[74,11,11,1528]],'ResNet base':[[3306,0,2,4],[2,1438,7,4],[6,19,327,4],[18,12,4,1590]],'RF GS':[[3161,2,100,49],[8,1396,34,13],[29,6,313,8],[92,8,107,1417]],'RL GS':[[3052,8,192,60],[9,1383,19,40],[27,13,310,6],[196,75,87,1266]],'ResNet 03':[[3309,0,1,2],[4,1438,7,2],[6,12,328,10],[33,31,2,1558]]}}

def scores(cm):
    cm=np.asarray(cm,dtype=float)
    tp=np.diag(cm); support=cm.sum(axis=1); predicted=cm.sum(axis=0)
    precision=np.divide(tp,predicted,out=np.zeros_like(tp),where=predicted>0)
    recall=np.divide(tp,support,out=np.zeros_like(tp),where=support>0)
    f1=np.divide(2*tp,support+predicted,out=np.zeros_like(tp),where=(support+predicted)>0)
    return dict(accuracy=tp.sum()/cm.sum(),f1_macro=f1.mean(),precision=precision,recall=recall,f1=f1,support=support)

def save(fig,out,name):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    fig.savefig(out/(name+'.png'),dpi=220,bbox_inches='tight')
    fig.savefig(out/(name+'.svg'),bbox_inches='tight')
    plt.close(fig)

def normalization(signal,out,name='01_normalizacion_real',fs=500):
    a=np.asarray(signal[:4000],dtype=np.float32)
    z=(a-a.mean())/(a.std()+1e-8)
    fig,axes=plt.subplots(1,2,figsize=(12,2.6),sharex=True)
    for ax,data,title,ylabel in zip(axes,[a,z],['Señal original','Normalización por ventana'],['Amplitud original','Amplitud normalizada']):
        ax.plot(np.arange(len(a))/fs,data,color=TEAL,lw=1)
        ax.set(title=title,xlabel='Tiempo (s)',ylabel=ylabel)
        ax.grid(alpha=.15)
    fig.tight_layout();save(fig,out,name)

def rr_features(signal,out,name='02_picos_rr_caracteristicas',fs=500):
    from scipy.signal import find_peaks
    a=np.asarray(signal,dtype=float)
    peaks,_=find_peaks(a,distance=150,prominence=.2)
    if len(peaks)<3: raise ValueError('Este registro no da suficientes picos con el detector de etapa 2.')
    rr=np.diff(peaks)/fs
    hr=60/rr.mean();rmssd=np.sqrt(np.mean(np.diff(rr)**2))
    fig,axes=plt.subplots(2,1,figsize=(8,5.4),gridspec_kw={'height_ratios':[1.5,1]})
    axes[0].plot(np.arange(len(a))/fs,a,lw=1,color=TEAL)
    axes[0].scatter(peaks/fs,a[peaks],s=20,color='#DB6E36',zorder=3,label='Picos detectados')
    axes[0].set(title='Extracción de intervalos RR',xlabel='Tiempo (s)',ylabel='Amplitud del registro')
    axes[0].legend(loc='upper right',fontsize=9)
    axes[1].plot(np.arange(1,len(rr)+1),rr,'o-',color=TEAL)
    axes[1].set(xlabel='Intervalo consecutivo',ylabel='RR (s)',title=f'HR estimada: {hr:.1f} lpm     RMSSD: {rmssd*1000:.1f} ms')
    axes[1].grid(alpha=.2)
    fig.tight_layout();save(fig,out,name)
    return dict(hr_lpm=float(hr),rmssd_ms=float(rmssd*1000),peaks=len(peaks))

def confusion_both(out):
    fig,axes=plt.subplots(1,2,figsize=(12,4.6),gridspec_kw={'width_ratios':[1,1.5]})
    for stage,ax in zip([1,2],axes):
        cm=np.asarray(MATRICES[stage]['ResNet base']);pct=cm/cm.sum(axis=1,keepdims=True)
        ax.imshow(pct,vmin=0,vmax=1,cmap='BuGn')
        for r in range(len(cm)):
            for c in range(len(cm)):
                ax.text(c,r,f'{cm[r,c]}\n{pct[r,c]:.1%}',ha='center',va='center',color='white' if pct[r,c]>.55 else INK,fontsize=11)
        ax.set(xticks=np.arange(len(cm)),yticks=np.arange(len(cm)),xticklabels=CLASSES[stage],yticklabels=CLASSES[stage],xlabel='Predicción',ylabel='Clase real',title=f'ResNet base, etapa {stage}')
        ax.tick_params(axis='x',labelrotation=20)
    fig.suptitle('Aciertos y errores relativos dentro de cada clase',fontsize=16,fontweight='bold')
    fig.tight_layout();save(fig,out,'03_matrices_resnet_base')

def class_metrics(out,stage=2):
    m=scores(MATRICES[stage]['ResNet base']);labels=CLASSES[stage]
    y=np.arange(len(labels));fig,ax=plt.subplots(figsize=(6.8,3.4))
    ax.barh(y-.18,m['precision'],height=.34,label='Precisión',color=TEAL)
    ax.barh(y+.18,m['recall'],height=.34,label='Recall',color=GOLD)
    for values,offset in [(m['precision'],-.18),(m['recall'],.18)]:
        for i,v in enumerate(values):ax.text(v+.01,i+offset,f'{v*100:.1f}%',va='center',fontsize=10)
    ax.set(yticks=y,yticklabels=labels,xlim=(0,1.16),title=f'ResNet base: métricas por clase, etapa {stage}')
    ax.invert_yaxis();ax.legend(loc='lower center',bbox_to_anchor=(.5,-.42),ncol=2,frameon=False)
    ax.set_xticks([0,.25,.5,.75,1],['0%','25%','50%','75%','100%'])
    fig.tight_layout();save(fig,out,f'04_metricas_por_clase_etapa{stage}')

def comparison(out):
    labels=list(MATRICES[1]);x=np.arange(len(labels));fig,ax=plt.subplots(figsize=(12,4.5))
    for stage,offset,color in [(1,-.19,TEAL),(2,.19,GOLD)]:
        values=[scores(MATRICES[stage][model])['f1_macro'] for model in labels]
        ax.bar(x+offset,values,width=.36,color=color,label=f'Etapa {stage}')
        for pos,value in zip(x+offset,values):ax.text(pos,value+.015,f'{value*100:.2f}',ha='center',fontsize=10)
    ax.set(xticks=x,xticklabels=labels,ylim=(0,1.1),ylabel='F1 macro de validación',title='Comparación de modelos con las mismas particiones')
    ax.set_yticks([0,.25,.5,.75,1],['0%','25%','50%','75%','100%'])
    ax.legend(ncol=2,loc='upper left',frameon=False)
    fig.tight_layout();save(fig,out,'05_comparacion_f1_precisa')

def imbalance(out):
    labels=['RL base','RF base','ResNet base'];x=np.arange(3);fig,ax=plt.subplots(figsize=(9,4))
    for key,offset,color,name in [('accuracy',-.19,TEAL,'Accuracy'),('f1_macro',.19,GOLD,'F1 macro')]:
        values=[scores(MATRICES[1][label])[key] for label in labels]
        ax.bar(x+offset,values,width=.36,color=color,label=name)
        for pos,value in zip(x+offset,values):ax.text(pos,value+.015,f'{value*100:.2f}%',ha='center',fontsize=11)
    ax.set(xticks=x,xticklabels=labels,ylim=(0,1.13),ylabel='Proporción',title='Etapa 1: 43 casos de reposo y 173 de actividad')
    ax.legend(frameon=False,ncol=2);fig.tight_layout();save(fig,out,'06_desbalance_accuracy_f1')

def static_figures(out):
    confusion_both(out);class_metrics(out,1);class_metrics(out,2);comparison(out);imbalance(out)


## Fuente de los resultados
Si existe un CSV de predicciones y su metadata corresponde a la versión preparada, se usa ese archivo. De lo contrario se mantienen las matrices de las imágenes originales. Si una corrida cambió, sus cifras pueden diferir de la guía de exposición.


In [ ]:
FULL_LABELS={1:['Reposo','Actividad'],2:['Bradicardia sinusal','Taquicardia sinusal','Fibrilación auricular','Ritmo sinusal normal']}
ARTIFACTS={'RL base':'baseline','RF base':'baseline_rf','ResNet base':'baseline_resnet','RF GS':'rf','RL GS':'lr','ResNet 03':'resnet'}
provenance=[]
for stage in STAGES:
    for model, artifact in ARTIFACTS.items():
        source='Matriz transcrita de ImagenesDeColab'
        filename=RESULTS / f'stage{stage}_{artifact}_predictions.csv'
        meta_file=MODELS / f'stage{stage}_{artifact}_meta.json'
        prepared=PREPARED / f'stage{stage}.npz'
        if USE_CURRENT_PREDICTIONS and filename.exists() and meta_file.exists() and prepared.exists():
            meta=json.loads(meta_file.read_text())
            with np.load(prepared,allow_pickle=False) as d:
                assert str(d['dataset_id'])==meta['dataset_id'], 'Modelos y datos no coinciden; repetir 02–04.'
            df=pd.read_csv(filename)
            mapping={label:i for i,label in enumerate(FULL_LABELS[stage])}
            assert df['real'].isin(mapping).all() and df['predicted'].isin(mapping).all()
            cm=np.zeros((len(mapping),len(mapping)),dtype=int)
            for real,pred in zip(df['real'].map(mapping),df['predicted'].map(mapping)):
                cm[real,pred]+=1
            MATRICES[stage][model]=cm.tolist()
            source=str(filename)
        provenance.append(dict(stage=stage,model=model,source=source))
pd.DataFrame(provenance).to_csv(OUT/'fuentes_figuras.csv',index=False)
display(pd.DataFrame(provenance))
static_figures(OUT)
rows=[]
for stage in STAGES:
    for model,cm in MATRICES[stage].items():
        m=scores(cm)
        rows.append(dict(stage=stage,model=model,accuracy=m['accuracy'],f1_macro=m['f1_macro']))
pd.DataFrame(rows).to_csv(OUT/'metricas_verificadas.csv',index=False)
display(pd.DataFrame(rows))
display(Image(filename=str(OUT/'03_matrices_resnet_base.png')))


## Señal real antes y después de normalizar
Muestra una ventana de entrenamiento, sin usarla para estimar desempeño. La operación resta media y divide por desviación poblacional + 1e-8. No es un filtro de ruido.


In [ ]:
filename=PREPARED/'stage1.npz'
if filename.exists():
    with np.load(filename,allow_pickle=False) as d:
        index=int(d['train_idx'][0])
        normalization(d['raw'][index],OUT)
        print('ID del ejemplo:',d['ids'][index])
    display(Image(filename=str(OUT/'01_normalizacion_real.png')))
else:
    print('Falta stage1.npz: ejecutar 01 con etapa 1 habilitada.')


## Picos detectados, intervalos RR y características
Se selecciona el primer registro de entrenamiento de clase normal que permita aplicar el mismo detector de etapa 2. Son estimaciones algorítmicas, no anotaciones médicas de picos R. Revisar visualmente falsos picos o latidos omitidos.


In [ ]:
filename=PREPARED/'stage2.npz'
if filename.exists():
    found=False
    with np.load(filename,allow_pickle=False) as d:
        indices=[int(i) for i in d['train_idx'] if d['y'][i]==3]
        raw=d['raw']
        for index in indices[:100]:
            try:
                values=rr_features(raw[index],OUT)
                print('ID del ejemplo:',d['ids'][index],values)
                found=True;break
            except ValueError:
                continue
    del raw;gc.collect()
    if found:display(Image(filename=str(OUT/'02_picos_rr_caracteristicas.png')))
    else:print('No se encontró ejemplo adecuado entre los primeros 100 registros normales de entrenamiento.')
else:
    print('Falta stage2.npz: ejecutar 01 con etapa 2 habilitada.')


## Distribución de HR estimada por clase
Ayuda a explicar las features para los modelos clásicos. No demuestra que una variable por sí sola clasifique todas las patologías. Se usa entrenamiento.


In [ ]:
filename=PREPARED/'stage2.npz'
if filename.exists():
    with np.load(filename,allow_pickle=False) as d:
        idx=d['train_idx'];features=d['features'];y=d['y']
    grouped=[features[idx][y[idx]==label,0] for label in range(4)]
    fig,ax=plt.subplots(figsize=(9,4))
    ax.boxplot(grouped,showfliers=False)
    ax.set_xticks(range(1,5),CLASSES[2])
    ax.set(ylabel='HR estimada (latidos/min)',title='HR extraída de los registros de entrenamiento')
    fig.tight_layout();save(fig,OUT,'07_hr_por_clase')
    display(Image(filename=str(OUT/'07_hr_por_clase.png')))
    del features;gc.collect()
else:print('Faltan los datos preparados de etapa 2.')


## Curvas de la base y de la corrida 03
Esta figura resuelve una limitación de la presentación: las curvas originales mostradas son de la corrida 03. Para comparar sobreajuste de la red base hay que usar sus propios historiales. No se deducen curvas a partir de métricas finales.


In [ ]:
for stage in STAGES:
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    plotted=0
    for ax,artifact,title in zip(axes,['baseline_resnet','resnet'],['ResNet base','ResNet cuaderno 03']):
        filename=RESULTS/f'stage{stage}_{artifact}_history.csv'
        meta_file=MODELS/f'stage{stage}_{artifact}_meta.json'
        if not filename.exists():
            ax.axis('off');ax.text(.1,.5,'Historial no disponible');continue
        history=pd.read_csv(filename)
        ax.plot(history['epoch'],history['train_loss'],label='Entrenamiento',color=TEAL)
        ax.plot(history['epoch'],history['val_loss'],label='Validación',color=GOLD)
        if meta_file.exists():
            meta=json.loads(meta_file.read_text())
            with np.load(PREPARED/f'stage{stage}.npz',allow_pickle=False) as d:
                assert str(d['dataset_id'])==meta['dataset_id'],'Historial de otra versión de datos'
            ax.axvline(meta['best_epoch'],ls='--',color='gray',label=f'Checkpoint {meta["best_epoch"]}')
        ax.set(title=title,xlabel='Época',ylabel='Pérdida');ax.legend();plotted+=1
    if plotted:
        fig.suptitle(f'Etapa {stage}: comparación de convergencia')
        fig.tight_layout();save(fig,OUT,f'08_curvas_base_vs_03_etapa{stage}')
        display(Image(filename=str(OUT/f'08_curvas_base_vs_03_etapa{stage}.png')))
    else:
        plt.close(fig);print(f'Etapa {stage}: recuperar historiales de 02 y 03 en Drive.')


## Mejores hiperparámetros y detalle de Grid Search
Los rangos de la diapositiva 6 no son los parámetros ganadores. Esta celda recupera los valores exactos de la corrida y muestra sus mejores combinaciones.


In [ ]:
best=[]
for stage in STAGES:
    for name in ['rf','lr']:
        meta_file=MODELS/f'stage{stage}_{name}_meta.json'
        if meta_file.exists():
            meta=json.loads(meta_file.read_text())
            best.append(dict(stage=stage,model=name,cv_f1_macro=meta.get('cv_f1_macro'),best_params=json.dumps(meta.get('best_params'),ensure_ascii=False)))
        else:print('No se encontró',meta_file)
        filename=RESULTS/f'stage{stage}_{name}_grid_search.csv'
        if filename.exists():
            results=pd.read_csv(filename).sort_values('rank_test_score')
            display(results[['params','mean_test_score','std_test_score','rank_test_score']].head(5))
if best:
    best=pd.DataFrame(best);best.to_csv(OUT/'mejores_hiperparametros.csv',index=False);display(best)


## Ejemplo acertado y ejemplo erróneo del modelo elegido
Se leen las predicciones de `baseline_resnet`, no las de la corrida 03. Se elige el primer acierto y el primer error disponibles; no se interpreta la confianza como certeza clínica.


In [ ]:
for stage in STAGES:
    pred_file=RESULTS/f'stage{stage}_baseline_resnet_predictions.csv'
    data_file=PREPARED/f'stage{stage}.npz'
    meta_file=MODELS/f'stage{stage}_baseline_resnet_meta.json'
    if not all(p.exists() for p in [pred_file,data_file,meta_file]):
        print(f'Etapa {stage}: faltan predicciones/base/metadatos; ejecutar 02 y 04.');continue
    predictions=pd.read_csv(pred_file,dtype={'id':str})
    correct=predictions['real']==predictions['predicted']
    selected=pd.concat([predictions[correct].head(1),predictions[~correct].head(1)])
    if selected.empty:continue
    with np.load(data_file,allow_pickle=False) as d:
        meta=json.loads(meta_file.read_text())
        assert str(d['dataset_id'])==meta['dataset_id']
        lookup={str(value):index for index,value in enumerate(d['ids'])}
        raw=d['raw']
    fig,axes=plt.subplots(len(selected),1,figsize=(11,3*len(selected)),squeeze=False)
    for ax,(_,row) in zip(axes.ravel(),selected.iterrows()):
        signal=raw[lookup[row['id']]]
        ax.plot(np.arange(len(signal))/500,signal,color=TEAL,lw=.8)
        ax.set(xlabel='Tiempo (s)',ylabel='Amplitud',title=f'Real: {row["real"]} / Predicción: {row["predicted"]} / Confianza: {row["confidence"]:.3f}')
    fig.tight_layout();save(fig,OUT,f'09_ejemplos_base_etapa{stage}')
    display(Image(filename=str(OUT/f'09_ejemplos_base_etapa{stage}.png')))
    del raw;gc.collect()


## Descargar figuras
Elegir las que aporten a la explicación y mantener las 10 diapositivas. PNG funciona para importar en Gamma/Canva; SVG permite mayor nitidez donde se admita. Conservar las fuentes y los CSV de procedencia.


In [ ]:
archive=Path('/content/visuales_defensa_Pulso_IA.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for filename in OUT.iterdir():
        if filename.is_file():z.write(filename,filename.name)
files.download(str(archive))
